# NER Powered Semantic Search Using Pinecone v5.0.0

### Setup Environment

In [1]:
# init pinecone

from pinecone import Pinecone, ServerlessSpec
# API_KEY = "YOUR API KEY"
# pc = Pinecone(api_key = "")

index = pc.Index("test2")

In [2]:
pc.list_indexes()

[
    {
        "name": "medium-data",
        "metric": "cosine",
        "host": "medium-data-kbs8m4f.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-east-1",
                "cloud": "aws",
                "read_capacity": {
                    "mode": "OnDemand",
                    "status": {
                        "state": "Ready",
                        "current_shards": null,
                        "current_replicas": null
                    }
                }
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 768,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "medium-datas",
        "metric": "cosine",
        "host": "medium-datas-kbs8m4f.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-east-1

In [28]:
# clean up pinecone index, after deleting all vectors if you run it again you will get error

# index.delete(delete_all=True)

In [29]:
# delete index , dimension no longer useful
# pc.delete_index("test")

In [3]:
# load libraries for NER 

# from transformers import AutoTokenizer, AutoModelForTokenClassification
# from transformers import pipeline
# import torch

from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
import torch


### NER Engine

In [4]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [5]:
# init NER engine

# model_id = 'dslim/bert-base-NER'

# tokenizer = AutoTokenizer.from_pretrained(model_id)
# model = AutoModelForTokenClassification.from_pretrained(model_id)

# # nlp pipeline

# nlp = pipeline('ner',
#               model=model,
#               tokenizer=tokenizer,
#               aggregation_strategy= 'max',
#               device= 'cpu') 
# # nlp("Bill Gates is the founder of Microsoft")


model_id = 'dslim/bert-base-NER'

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForTokenClassification.from_pretrained(model_id)

nlp = pipeline('ner',
              model=model,
              tokenizer=tokenizer,
              aggregation_strategy='max',
              device='cpu')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Retriever

In [6]:
# !pip install --upgrade pyarrow
# !pip install pyarrow==15.0.2 datasets==2.19.1 sentence-transformers
import pyarrow as pa
print(pa.__version__)

15.0.2


In [7]:
# load libraries for retriever

# from sentence_transformers import SentenceTransformer


# # https://huggingface.co/flax-sentence-embeddings/all_datasets_v3_mpnet-base
# retriever = SentenceTransformer(
#     "flax-sentence-embeddings/all_datasets_v3_mpnet-base")


from sentence_transformers import SentenceTransformer

retriever = SentenceTransformer(
    "flax-sentence-embeddings/all_datasets_v3_mpnet-base"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
retriever

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [36]:
# Create Pinecone Index
pc.create_index("medium-datas", dimension= 768, metric="cosine",
                     spec=ServerlessSpec(cloud="aws", region="us-east-1"))


{
    "name": "medium-datas",
    "metric": "cosine",
    "host": "medium-datas-kbs8m4f.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 768,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "20

In [9]:
index= pc.Index("medium-datas")

### Data Prep

In [10]:
# !pip uninstall pyarrow -y
# !pip install pyarrow==15.0.2

In [10]:
import pyarrow
import pyarrow.dataset as pds
print(pyarrow.__version__)
print("pyarrow dataset works")

15.0.2
pyarrow dataset works


In [11]:

# !pip install datasets
from datasets import load_dataset

In [13]:
# Obtain Raw Data

# df = load_dataset(
#     "fabiochiu/medium-articles",
#     data_files="medium_articles.csv",
#     split="train"
# ).to_pandas()

# df = df.dropna().sample(10000, random_state=45) # might take 30mins to 1hr

# df['text_extended'] = df['title'] + '.' + df['text'].str[:1000]

df = load_dataset(
    "fabiochiu/medium-articles",
    data_files="medium_articles.csv",
    split="train"
).to_pandas()

df = df.dropna().sample(10000, random_state=45)

df['text_extended']= df['title'] +'.'+df['text'].str[:1000]


In [14]:
# Incase your internet is slow and couldn't make "dataset" works, you can download the file I uploaded as "medium_articles_10k.csv"
# Source of data: https://www.kaggle.com/code/fabiochiusano/medium-articles-simple-data-analysis?select=medium_articles.csv
# it is the same underlying data

# import pandas as pd
# df = pd.read_csv("medium_articles_10k.csv")
# df = df.dropna().sample(1000, random_state=45) # 
# df['text_extended'] = df['title'] + '.' + df['text'].str[:1000]

import pandas as pd

df = pd.read_csv("medium_articles_10k.csv")
df = df.dropna().sample(1000, random_state=45)
df['text_extended'] = df['title']+'.'+df['text'].str[:1000]

In [15]:
df = df.iloc[0:10000]

In [16]:
df.head()

,Unnamed: 0,title,text,url,authors,timestamp,tags,text_extended
3577,3577,Why You’re Struggling with Innovation. And How...,The modern typewriter had a problem. When Chri...,https://jswilder16.medium.com/why-youre-strugg...,['Jake Wilder'],2019-04-22 01:46:42.469000+00:00,"['Management', 'Leadership', 'Innovation', 'Cr...",Why You’re Struggling with Innovation. And How...
7003,7003,#085 | Debugging Vue Sites Displaying Blank Pages,So I had this bright idea to consolidate all m...,https://medium.com/footprints-on-the-sand/085-...,['Kelvin Zhao'],2020-03-24 23:11:00.960000+00:00,"['Vuejs', 'Website Development', 'Debugging', ...",#085 | Debugging Vue Sites Displaying Blank Pa...
925,925,You won’t believe what I’m going to tell you a...,You won’t believe what I’m going to tell you a...,https://medium.com/pnewton84/he-discovered-a-n...,['Paul Newton'],2017-09-19 19:26:41.376000+00:00,"['Marketing', 'Digital Marketing', 'Facebook',...",You won’t believe what I’m going to tell you a...
3233,3233,How Emotionally Intelligent People Deal With T...,This reminds me of the time I was on the way h...,https://medium.com/curious/how-emotionally-int...,['Ayodeji Awosika'],2020-11-09 20:50:40.841000+00:00,"['Emotional Intelligence', 'Mental Health', 'I...",How Emotionally Intelligent People Deal With T...
3515,3515,What’s New in React 16 and Fiber Explanation,"Previously, React would block the entire threa...",https://medium.com/edge-coders/react-16-featur...,['Trey Huffine'],2018-10-30 13:57:05.603000+00:00,"['Code', 'Tech', 'React', 'JavaScript', 'Start...",What’s New in React 16 and Fiber Explanation.P...


In [17]:
df_batch = df['text_extended'].iloc[0]
df_batch

'Why You’re Struggling with Innovation. And How to Get Better..The modern typewriter had a problem. When Christopher Sholes developed the first model in 1868, it was an amazing development for its time. But if you tried to type too quickly on it, the type bars had a tendency to bang into one another and get stuck.\n\nSholes consulted an educator who helped him analyze the most common pairings of letters. He then split up those letters so that their type bars were farther apart and less likely to jam.\n\nHe slowed down typing speed to prevent the typewriter from jamming. Which then sped up the typing.\n\nThis dictated the layout of the keyboard, which came to be known for the first six letters in the upper row — QWERTY.\n\nIn 1873, the Sholes & Glidden typewriter became the first to be mass-produced, and its keyboard layout soon became standard. Nearly 150 years later, despite the fact that sticking type bars are an obsolete problem, the original inefficient design continues to be the s

In [18]:
 [item['word'] for item in nlp(df_batch)]# list of entities

['Christopher Sholes', 'Sholes', 'QWERTY', 'Sholes & Glidden']

In [19]:
df_batchs = df['text_extended'].iloc[0:4].tolist()
df_batchs

['Why You’re Struggling with Innovation. And How to Get Better..The modern typewriter had a problem. When Christopher Sholes developed the first model in 1868, it was an amazing development for its time. But if you tried to type too quickly on it, the type bars had a tendency to bang into one another and get stuck.\n\nSholes consulted an educator who helped him analyze the most common pairings of letters. He then split up those letters so that their type bars were farther apart and less likely to jam.\n\nHe slowed down typing speed to prevent the typewriter from jamming. Which then sped up the typing.\n\nThis dictated the layout of the keyboard, which came to be known for the first six letters in the upper row — QWERTY.\n\nIn 1873, the Sholes & Glidden typewriter became the first to be mass-produced, and its keyboard layout soon became standard. Nearly 150 years later, despite the fact that sticking type bars are an obsolete problem, the original inefficient design continues to be the 

In [20]:
entities=[]
for doc in df_batchs:
    entities.append([item['word'] for item in nlp(doc)])

print(entities)

[['Christopher Sholes', 'Sholes', 'QWERTY', 'Sholes & Glidden'], [], [], []]


In [21]:
nlp(df_batchs[0])

[{'entity_group': 'PER',
  'score': 0.99902153,
  'word': 'Christopher Sholes',
  'start': 104,
  'end': 122},
 {'entity_group': 'PER',
  'score': 0.9985745,
  'word': 'Sholes',
  'start': 316,
  'end': 322},
 {'entity_group': 'MISC',
  'score': 0.90234596,
  'word': 'QWERTY',
  'start': 714,
  'end': 720},
 {'entity_group': 'ORG',
  'score': 0.98406315,
  'word': 'Sholes & Glidden',
  'start': 736,
  'end': 752}]

In [22]:
len(nlp(df_batch)) # list of lst

4

In [23]:
len(nlp(df_batchs))

4

### NER Helper Function

In [24]:
# helper function for extracting entities of a batch of texts

def extract_entities(list_of_text):
    entities = []
    for doc in list_of_text: 
        entities.append([item['word'] for item in nlp(doc)])
        # list of entities for 1 doc
    return entities

In [25]:
# embedding

len(retriever.encode(df_batch))

768

In [26]:
retriever.encode(df_batch).tolist()

[0.05938209220767021,
 -0.05587984621524811,
 -0.00025684060528874397,
 -0.022576093673706055,
 -0.022538116201758385,
 0.014586560428142548,
 0.05134730786085129,
 0.01942661963403225,
 0.0004563210532069206,
 0.005636707413941622,
 0.01094143558293581,
 -0.025676017627120018,
 -0.0052114748395979404,
 0.07836558669805527,
 -0.02873675525188446,
 0.010479453951120377,
 0.007388378493487835,
 0.015391374938189983,
 -0.003297272836789489,
 0.03341612219810486,
 0.03883333131670952,
 -0.009255506098270416,
 -0.029266050085425377,
 0.007247708272188902,
 0.03285254165530205,
 0.0033398510422557592,
 0.008476488292217255,
 -0.03049289621412754,
 -0.03677905350923538,
 0.008160448633134365,
 -0.042663950473070145,
 0.07858201861381531,
 -0.029105117544531822,
 -0.001153979916125536,
 3.3559519607706534e-08,
 -0.01684730127453804,
 0.022535035386681557,
 0.00914553739130497,
 -0.049101125448942184,
 0.059200216084718704,
 0.032806314527988434,
 0.041669685393571854,
 -0.0028220387175679207,


In [27]:
len(retriever.encode(df_batch[0])) # try for one doc embedding for batch


768

In [28]:
emb = retriever.encode(df_batch).tolist() # array to python list
print(emb)

[0.05938209220767021, -0.05587984621524811, -0.00025684060528874397, -0.022576093673706055, -0.022538116201758385, 0.014586560428142548, 0.05134730786085129, 0.01942661963403225, 0.0004563210532069206, 0.005636707413941622, 0.01094143558293581, -0.025676017627120018, -0.0052114748395979404, 0.07836558669805527, -0.02873675525188446, 0.010479453951120377, 0.007388378493487835, 0.015391374938189983, -0.003297272836789489, 0.03341612219810486, 0.03883333131670952, -0.009255506098270416, -0.029266050085425377, 0.007247708272188902, 0.03285254165530205, 0.0033398510422557592, 0.008476488292217255, -0.03049289621412754, -0.03677905350923538, 0.008160448633134365, -0.042663950473070145, 0.07858201861381531, -0.029105117544531822, -0.001153979916125536, 3.3559519607706534e-08, -0.01684730127453804, 0.022535035386681557, 0.00914553739130497, -0.049101125448942184, 0.059200216084718704, 0.032806314527988434, 0.041669685393571854, -0.0028220387175679207, 0.07893193513154984, 0.011920992285013199,

### Batch Upsert

In [30]:
# upsert data

# from tqdm.auto import tqdm

from tqdm.auto import tqdm

# batch_size = 64

# for i in range(0, len(df), batch_size):
#     i_end = min(i+batch_size, len(df))
#     # print(i, i_end) # starting and ending index of each batch

batch_size = 64
for i in range(0, len(df), batch_size):
    i_end = min(i+batch_size, len(df))
    print(i, i_end)  # starting and ending index of each batch
    
    # get a batch of data
    # df_batch = df.iloc[i: i_end].copy()
    df_batch = df.iloc[i: i_end].copy()
    
    # embedding
    # emb = retriever.encode(df_batch['text_extended'].tolist()
    #                       ).tolist() # array to python list
    emb = retriever.encode(df_batch['text_extended'].tolist()
                          ).tolist()
    
    # ner extraction
    # entities = extract_entities(df_batch['text_extended'].tolist())
    entities = extract_entities(df_batch['text_extended'].tolist())
    
    # [[]] --> [set1, set2, ], remove duplicate entities    
    # df_batch['named_entity'] = [list(set(entity)) for entity in entities] # one list per document

    df_batch['named_entity'] = [list(set(entity)) for entity in entities]
    # create meta data
    # df_batch = df_batch.drop('text', axis=1)
    df_batch = df_batch.drop('text', axis=1)
    
    # meta_data = df_batch.to_dict(orient='records') # pd.df to dictionary

    meta_data = df_batch.to_dict(orient ='records')
    # create ids
    
    # ids = [f"{idx}" for idx in range(i, i_end)] #

    ids = [f"{idx}" for idx in range(i, i_end)]
    # upsert
    
    # vectors_to_upsert = list(zip(ids, emb, meta_data))  # nd array to python list

    vectors_to_upsert = list(zip(ids, emb, meta_data))
    # _ = index.upsert(vectors= vectors_to_upsert)  
    
    _ = index.upsert(vectors = vectors_to_upsert)



0 64
64 128
128 192
192 256
256 320
320 384
384 448
448 512
512 576
576 640
640 704
704 768
768 832
832 896
896 960
960 1000


In [31]:
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '187',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 04 May 2026 10:28:01 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '44',
                                    'x-pinecone-request-latency-ms': '44',
                                    'x-pinecone-response-duration-ms': '45'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 1000}},
 'storageFullness': 0.0,
 'total_vector_count': 1000,
 'vector_type': 'dense'}

### Query data

In [35]:
# query = "How to make a Wordpress website?"  # Natural Language

# emb_qx = retriever.encode(query).tolist() # embedded query vector

# ne = extract_entities([query])[0]# Named entity as a search filter


query = "how to make a wordpress website?"
emb_qx = retriever.encode(query).tolist()
ne = extract_entities([query])[0]

In [36]:
# xc = index.query(vector=emb_qx, top_k= 5, include_metadata=True,
#            filter = {"named_entity": {"$in" : ne}})

xc = index.query(vector=emb_qx, top_k = 5, include_metadata=True,
                filter = {"named_entity":{"$in": ne}})

In [43]:
# you might not find any match if you are only upserting 1k data because of insufficient data there might not be good match, 
# try to load more data or tweak query based on data (glance over pinecone console and look for text_extended field in your vectors)
# for result in xc['matches']:
#     print(result['score'], " ", result['metadata']['named_entity'])


for result in xc['matches']:
    print(result['score'], " ", result['metadata']['named_entity'])

0.283930779   ['NLP', 'Allen NLP', 'PyTorch', 'Allen']
0.26279068   ['Harnham', 'Data', 'Datatech Analytics', 'NLP', 'London', 'LDA', 'Python', 'Science']
0.242308617   ['Lambda Layer', 'NLP']
0.237251282   ['It', 'Text Mining', 'The Shining', 'Stephen King', 'King', 'NLP', 'Carrie', 'Under the Dome', 'The Dark Tower']
0.211456299   ['Your Weekly AI', 'NLP', 'Machine Learning and Data Science', 'AI', 'ML']


In [42]:
# query = "How to learn NLP?"  # Natural Language

# emb_qx = retriever.encode(query).tolist() # embedded query vector

# ne = extract_entities([query])[0] # Named entity as a search filter

# xc = index.query(vector=emb_qx, top_k= 5, include_metadata=True,
#            filter = {"named_entity": {"$in" : ne}  })


query = "how to learn NLP?"
emb_qx = retriever.encode(query).tolist()
ne = extract_entities([query])[0]
xc = index.query(vector=emb_qx, top_k=5, include_metadata=True,
filter = {"named_entity":{"$in":ne}})

In [44]:
# for result in xc['matches']:
#     print(result['score'], " ", result['metadata']['named_entity'])


for result in xc['matches']:
    print(result['score'], " ", result['metadata']['named_entity'])

0.283930779   ['NLP', 'Allen NLP', 'PyTorch', 'Allen']
0.26279068   ['Harnham', 'Data', 'Datatech Analytics', 'NLP', 'London', 'LDA', 'Python', 'Science']
0.242308617   ['Lambda Layer', 'NLP']
0.237251282   ['It', 'Text Mining', 'The Shining', 'Stephen King', 'King', 'NLP', 'Carrie', 'Under the Dome', 'The Dark Tower']
0.211456299   ['Your Weekly AI', 'NLP', 'Machine Learning and Data Science', 'AI', 'ML']
